In [ ]:
# ============================================================
# 04_final_missingness_analysis
# Thesis final pipeline
# ============================================================

import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

from pathlib import Path
from scipy.stats import chi2_contingency, spearmanr, kruskal

pd.set_option("display.max_columns", 300)
pd.set_option("display.max_rows", 300)

PROJECT_CONSTANTS_DIR = Path(
    "/home/azureuser/cloudfiles/code/Users/m.bolognini/"
    "UseCase-Code/data-science/projects/a_MB_thesis_final/"
    "pipeline_thesis_final"
)

sys.path.append(str(PROJECT_CONSTANTS_DIR))

from project_constants import *

print("OUTPUT_01:", OUTPUT_01)
print("OUTPUT_03:", OUTPUT_03)
print("OUTPUT_04:", OUTPUT_04)

In [ ]:
# ============================================================
# LOAD DATASETS
# ============================================================

df_model = pd.read_parquet(
    OUTPUT_03 / "03_model_features.parquet"
)

df_daily = pd.read_parquet(
    OUTPUT_03 / "03_daily_labs.parquet"
)

df_episode_master = pd.read_parquet(
    OUTPUT_01 / "01_episode_master.parquet"
)

print("Model dataset:", df_model.shape)
print("Daily labs:", df_daily.shape)
print("Episode master:", df_episode_master.shape)

print("\nPatients:", df_model["pubID"].nunique())
print("Episodes:", df_model["episode_key"].nunique())

In [ ]:
df_episode_master.head()

In [ ]:
# ============================================================
# MERGE EPISODE COMPLEXITY VARIABLES FROM 01_EPISODE_MASTER
# ============================================================

for df in [df_model, df_daily, df_episode_master]:
    if "episode_key" in df.columns:
        df["episode_key"] = (
            df["episode_key"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

episode_complexity_cols = [
    "episode_key",

    # hospitalization complexity
    "LOS_days",
    "daysInER",

    # admission severity 
    "adm_SBP",	
    "adm_DBP",	
    "adm_SpO2",	
    "adm_FiO2",	
    "adm_heartRate",
    "adm_respiratoryRate",	
    "adm_PaO2",	
    "adm_bodyTemp",
     
    # frailty / function
    "adm_mews",
    "adm_gcsScore",
    "admECOG",
    "badm_barthelScore",
    "admBradenScore",
    "adm_dysphagia",

    # comorbidity burden
    "cirs_totalScore",
    "ccsi",

    # care pathway / administrative complexity
    "hospitalWard",
    "origin",
    "admissNumb6m",
    "adm_reason",

    # interventions
    "admc_antimicrIntensity",	
    "int_oxygenTherapy",	
    "int_dialysis",	
    "int_bloodTransfusion",	
    "int_vasoactiveDrug",	
    "nems_vitalParamMonitoring",	
    "admc_nosocomialInfection",
    "admc_diagnostics",

    # oncology
    "solidTumor_cat",

    # outcomes / discharge
    "dischargedAlive",
    "dischargeMod",
]

episode_complexity_cols = [
    c for c in episode_complexity_cols
    if c in df_episode_master.columns
]

episode_complexity = (
    df_episode_master[episode_complexity_cols]
    .drop_duplicates(subset=["episode_key"])
    .copy()
)

print("Episode complexity variables selected:")
print(episode_complexity.columns.tolist())

df = df_model.merge(
    episode_complexity,
    on="episode_key",
    how="left",
    suffixes=("", "_episode")
)

print("Dataset after episode complexity merge:", df.shape)
print("Duplicated episode_key:", df["episode_key"].duplicated().sum())

In [ ]:
# ============================================================
# FEATURE GROUPS
# ============================================================

mean_cols = [
    c for c in df.columns
    if "_mean_" in c
]

delta_cols = [
    c for c in df.columns
    if "_delta_" in c
]

count_cols = [
    c for c in df.columns
    if "_n_days_" in c
]

dynamic_cols = [
    c for c in df.columns
    if "_has_dynamic_" in c
]

lab_intensity_cols = [
    c for c in [
        "n_total_lab_records_0_7",
        "n_total_lab_days_0_7",
        "n_biomarkers_measured_0_7",
        "has_any_lab_0_7",
        "labs_per_day_0_7",
        "n_labs_day0_1",
    ]
    if c in df.columns
]

lab_feature_cols = (
    mean_cols
    + delta_cols
    + count_cols
    + dynamic_cols
    + lab_intensity_cols
)

print("Mean features:", len(mean_cols))
print("Delta features:", len(delta_cols))
print("Daily observation count features:", len(count_cols))
print("Dynamic indicators:", len(dynamic_cols))
print("Lab intensity features:", len(lab_intensity_cols))
print("Total laboratory features:", len(lab_feature_cols))

In [ ]:
# ============================================================
# OVERALL LABORATORY FEATURE MISSINGNESS
# ============================================================

overall_missingness = []

for col in lab_feature_cols:

    overall_missingness.append({
        "feature": col,
        "missing_n": df[col].isna().sum(),
        "missing_pct": 100 * df[col].isna().mean(),
        "available_n": df[col].notna().sum(),
    })

overall_missingness = (
    pd.DataFrame(overall_missingness)
    .sort_values("missing_pct", ascending=False)
)

display(overall_missingness.head(40))

In [ ]:
# ============================================================
# TOP MISSING FEATURES VISUALIZATION
# ============================================================

top_missing = overall_missingness.head(25).copy()

plt.figure(figsize=(10, 8))

sns.barplot(
    data=top_missing,
    y="feature",
    x="missing_pct"
)

plt.xlabel("Missing percentage")
plt.ylabel("Feature")
plt.title("Top 25 laboratory features by missingness")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# LAB AVAILABILITY HEATMAP BY DAY
# ============================================================

heatmap_df = (
    df_daily
    .groupby(["biomarker", "lab_day"])["episode_key"]
    .nunique()
    .reset_index()
)

heatmap_pivot = heatmap_df.pivot(
    index="biomarker",
    columns="lab_day",
    values="episode_key"
)

plt.figure(figsize=(12, 6))

sns.heatmap(
    heatmap_pivot,
    annot=True,
    fmt=".0f",
    cmap="Blues"
)

plt.title("Number of contributing episodes by biomarker and hospitalization day")
plt.xlabel("Hospitalization day")
plt.ylabel("Biomarker")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MISSINGNESS BY COMPOSITE OUTCOME
# Composite: in-hospital mortality OR discharge to palliative care
# ============================================================

rows = []

for col in mean_cols + delta_cols:

    for outcome in [0, 1]:

        tmp = df[
            df["composite_inhospital_or_palliative"] == outcome
        ]

        rows.append({
            "feature": col,
            "outcome": outcome,
            "missing_pct": 100 * tmp[col].isna().mean(),
        })

missing_by_composite = pd.DataFrame(rows)

pivot_missing_by_composite = (
    missing_by_composite
    .pivot(
        index="feature",
        columns="outcome",
        values="missing_pct"
    )
)

pivot_missing_by_composite.columns = [
    "Composite_0_missing_pct",
    "Composite_1_missing_pct"
]

display(
    pivot_missing_by_composite.sort_values(
        "Composite_1_missing_pct",
        ascending=False
    ).head(30)
)

In [ ]:
# ============================================================
# MISSINGNESS BY 90-DAY POST-DISCHARGE MORTALITY
# ============================================================

rows = []

for col in mean_cols + delta_cols:

    for outcome in [0, 1]:

        tmp = df[
            df["mortality_90d_post_discharge"] == outcome
        ]

        rows.append({
            "feature": col,
            "outcome": outcome,
            "missing_pct": 100 * tmp[col].isna().mean(),
        })

missing_by_90d = pd.DataFrame(rows)

pivot_missing_by_90d = (
    missing_by_90d
    .pivot(
        index="feature",
        columns="outcome",
        values="missing_pct"
    )
)

pivot_missing_by_90d.columns = [
    "Mortality90d_0_missing_pct",
    "Mortality90d_1_missing_pct"
]

display(
    pivot_missing_by_90d.sort_values(
        "Mortality90d_1_missing_pct",
        ascending=False
    ).head(30)
)

In [ ]:
# ============================================================
# MISSINGNESS SIGNIFICANCE TESTING
# COMPOSITE OUTCOME
# Exploratory MNAR screening with FDR correction
# ============================================================

rows = []

for col in mean_cols + delta_cols:

    missing_flag = df[col].isna().astype(int)

    valid = df["composite_inhospital_or_palliative"].notna()

    contingency = pd.crosstab(
        missing_flag[valid],
        df.loc[valid, "composite_inhospital_or_palliative"]
    )

    try:
        _, p, _, _ = chi2_contingency(contingency)
    except Exception:
        p = np.nan

    rows.append({
        "feature": col,
        "missing_pct": 100 * missing_flag.mean(),
        "p_value_vs_composite": p,
    })

missing_tests_composite = pd.DataFrame(rows)

# Nominal unadjusted significance
missing_tests_composite["significant_nominal"] = (
    missing_tests_composite["p_value_vs_composite"] < 0.05
)

# FDR-BH correction
missing_tests_composite["p_value_fdr"] = np.nan
missing_tests_composite["significant_fdr"] = False

valid_p = missing_tests_composite["p_value_vs_composite"].notna()

if valid_p.sum() > 0:

    reject, pvals_fdr, _, _ = multipletests(
        missing_tests_composite.loc[valid_p, "p_value_vs_composite"],
        method="fdr_bh"
    )

    missing_tests_composite.loc[valid_p, "p_value_fdr"] = pvals_fdr
    missing_tests_composite.loc[valid_p, "significant_fdr"] = reject

display(
    missing_tests_composite
    .sort_values("p_value_fdr")
    .head(30)
)

print(
    "Nominally significant features:",
    missing_tests_composite["significant_nominal"].sum(),
    "/",
    len(missing_tests_composite)
)

print(
    "FDR-significant features:",
    missing_tests_composite["significant_fdr"].sum(),
    "/",
    len(missing_tests_composite)
)

In [ ]:
# ============================================================
# MISSINGNESS SIGNIFICANCE TESTING
# 90-DAY POST-DISCHARGE MORTALITY
# Exploratory MNAR screening with FDR correction
# ============================================================

rows = []

for col in mean_cols + delta_cols:

    missing_flag = df[col].isna().astype(int)

    valid = df["mortality_90d_post_discharge"].notna()

    contingency = pd.crosstab(
        missing_flag[valid],
        df.loc[valid, "mortality_90d_post_discharge"]
    )

    try:
        _, p, _, _ = chi2_contingency(contingency)
    except Exception:
        p = np.nan

    rows.append({
        "feature": col,
        "missing_pct": 100 * missing_flag.mean(),
        "p_value_vs_90d_mortality": p,
    })

missing_tests_90d = pd.DataFrame(rows)

# Nominal unadjusted significance
missing_tests_90d["significant_nominal"] = (
    missing_tests_90d["p_value_vs_90d_mortality"] < 0.05
)

# FDR-BH correction
missing_tests_90d["p_value_fdr"] = np.nan
missing_tests_90d["significant_fdr"] = False

valid_p = missing_tests_90d["p_value_vs_90d_mortality"].notna()

if valid_p.sum() > 0:

    reject, pvals_fdr, _, _ = multipletests(
        missing_tests_90d.loc[valid_p, "p_value_vs_90d_mortality"],
        method="fdr_bh"
    )

    missing_tests_90d.loc[valid_p, "p_value_fdr"] = pvals_fdr
    missing_tests_90d.loc[valid_p, "significant_fdr"] = reject

display(
    missing_tests_90d
    .sort_values("p_value_fdr")
    .head(30)
)

print(
    "Nominally significant features:",
    missing_tests_90d["significant_nominal"].sum(),
    "/",
    len(missing_tests_90d)
)

print(
    "FDR-significant features:",
    missing_tests_90d["significant_fdr"].sum(),
    "/",
    len(missing_tests_90d)
)

In [ ]:
# ============================================================
# MONITORING INTENSITY VARIABLES
# ============================================================

monitoring_cols = [
    "n_total_lab_records_0_7",
    "n_total_lab_days_0_7",
    "n_biomarkers_measured_0_7",
    "labs_per_day_0_7",
    "n_labs_day0_1",
]

monitoring_cols = [
    c for c in monitoring_cols
    if c in df.columns
]

print("Monitoring intensity variables:")
print(monitoring_cols)

display(
    df[monitoring_cols].describe().T
)

In [ ]:
# ============================================================
# EPISODE COMPLEXITY VARIABLES
# ============================================================

episode_complexity_vars = [
    "episode_key",

    # hospitalization complexity
    "LOS_days",
    "daysInER",

    # admission severity 
    "adm_SBP",	
    "adm_DBP",	
    "adm_SpO2",	
    "adm_FiO2",	
    "adm_heartRate",
    "adm_respiratoryRate",	
    "adm_PaO2",	
    "adm_bodyTemp",
     
    # frailty / function
    "adm_mews",
    "adm_gcsScore",
    "admECOG",
    "badm_barthelScore",
    "admBradenScore",
    "adm_dysphagia",

    # comorbidity burden
    "cirs_totalScore",
    "ccsi",

    # care pathway / administrative complexity
    "hospitalWard",
    "origin",
    "admissNumb6m",
    "adm_reason",

    # interventions
    "admc_antimicrIntensity",	
    "int_oxygenTherapy",	
    "int_dialysis",	
    "int_bloodTransfusion",	
    "int_vasoactiveDrug",	
    "nems_vitalParamMonitoring",	
    "admc_nosocomialInfection",
    "admc_diagnostics",

    # oncology
    "solidTumor_cat",

    # outcomes / discharge
    "dischargedAlive",
    "dischargeMod",
]

episode_complexity_vars = [
    c for c in episode_complexity_vars
    if c in df.columns
]

print("Episode complexity variables:")
print(episode_complexity_vars)

display(
    df[episode_complexity_vars].describe().T
)

In [ ]:
# ============================================================
# SPEARMAN CORRELATION:
# MONITORING INTENSITY VS EPISODE COMPLEXITY
# ============================================================

corr_rows = []

for monitoring_var in monitoring_cols:

    for complexity_var in episode_complexity_vars:

        tmp = df[[monitoring_var, complexity_var]].copy()
        tmp[monitoring_var] = pd.to_numeric(tmp[monitoring_var], errors="coerce")
        tmp[complexity_var] = pd.to_numeric(tmp[complexity_var], errors="coerce")
        tmp = tmp.dropna()

        if tmp.shape[0] < 20:
            rho = np.nan
            p = np.nan
        else:
            rho, p = spearmanr(
                tmp[monitoring_var],
                tmp[complexity_var]
            )

        corr_rows.append({
            "monitoring_variable": monitoring_var,
            "complexity_variable": complexity_var,
            "spearman_rho": rho,
            "p_value": p,
            "n": tmp.shape[0],
        })

monitoring_complexity_corr = pd.DataFrame(corr_rows)

display(
    monitoring_complexity_corr
    .sort_values("spearman_rho", ascending=False)
    .head(40)
)

In [ ]:
# ============================================================
# MONITORING INTENSITY QUARTILES
# ============================================================

df["lab_monitoring_quartile"] = pd.qcut(
    df["n_total_lab_records_0_7"],
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
    duplicates="drop"
)

print(df["lab_monitoring_quartile"].value_counts(dropna=False))

In [ ]:
# ============================================================
# EPISODE COMPLEXITY ACROSS MONITORING QUARTILES
# ============================================================

quartile_summary_rows = []

for var in episode_complexity_vars:

    tmp = df[["lab_monitoring_quartile", var]].copy()
    tmp[var] = pd.to_numeric(tmp[var], errors="coerce")

    summary = (
        tmp
        .groupby("lab_monitoring_quartile", observed=True)[var]
        .agg(
            n="count",
            median="median",
            q1=lambda x: x.quantile(0.25),
            q3=lambda x: x.quantile(0.75),
            mean="mean",
        )
        .reset_index()
    )

    # Kruskal-Wallis test
    groups = [
        g[var].dropna()
        for _, g in tmp.groupby("lab_monitoring_quartile", observed=True)
    ]

    groups = [g for g in groups if len(g) > 5]

    try:
        _, p = kruskal(*groups)
    except Exception:
        p = np.nan

    summary["variable"] = var
    summary["kruskal_p_value"] = p

    quartile_summary_rows.append(summary)

monitoring_quartile_summary = pd.concat(
    quartile_summary_rows,
    ignore_index=True
)

display(
    monitoring_quartile_summary[
        [
            "variable",
            "lab_monitoring_quartile",
            "n",
            "median",
            "q1",
            "q3",
            "mean",
            "kruskal_p_value",
        ]
    ]
)

In [ ]:
# ============================================================
# COMPACT MONITORING QUARTILE TABLE
# ============================================================


key_complexity_vars = [
    

    # hospitalization complexity
    "LOS_days",
    "daysInER",
     
    # frailty / function
    "adm_mews",
    "admECOG",
    "badm_barthelScore",
    "admBradenScore",
   

    # comorbidity burden
    "cirs_totalScore",
    "ccsi",

    # interventions
    "admc_antimicrIntensity",	
    "int_oxygenTherapy",	
    "int_dialysis",	
    "int_bloodTransfusion",	
    "int_vasoactiveDrug",	
    "nems_vitalParamMonitoring",	
    "admc_nosocomialInfection",
    "admc_diagnostics",

    # outcomes
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
]


key_complexity_vars = [
    c for c in key_complexity_vars
    if c in df.columns
]

compact_rows = []

for var in key_complexity_vars:

    row = {
        "variable": var
    }

    for q in df["lab_monitoring_quartile"].dropna().cat.categories:

        s = pd.to_numeric(
            df.loc[df["lab_monitoring_quartile"] == q, var],
            errors="coerce"
        ).dropna()

        if len(s) == 0:
            row[str(q)] = "NA"
        else:
            row[str(q)] = (
                f"{s.median():.1f} "
                f"({s.quantile(0.25):.1f}–{s.quantile(0.75):.1f})"
            )

    compact_rows.append(row)

monitoring_quartile_compact = pd.DataFrame(compact_rows)

display(monitoring_quartile_compact)

In [ ]:
# ============================================================
# PLOT: LENGTH OF STAY BY MONITORING QUARTILE
# ============================================================

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="lab_monitoring_quartile",
    y="LOS_days"
)

plt.xlabel("Laboratory monitoring intensity quartile")
plt.ylabel("Length of stay (days)")
plt.title("Length of stay by laboratory monitoring intensity")

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT: ANTI-MICROBIAL INTENSITY BY MONITORING QUARTILE
# ============================================================

plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x="lab_monitoring_quartile",
    y="admc_antimicrIntensity"
)

plt.xlabel("Laboratory monitoring intensity quartile")
plt.ylabel("Anti-microbial intensity")
plt.title("Anti-microbial intensity by laboratory monitoring intensity")

plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT: DIAGNOSTIC PROCEDURES BY MONITORING QUARTILE
# ============================================================

diagnostic_map = {
    "1": "First-line imaging",
    "2": "Advanced imaging",
    "3": "Invasive procedures",
    "4": "Daily blood tests at least once",
}

diag = df[
    ["episode_key", "lab_monitoring_quartile", "admc_diagnostics"]
].copy()

diag["admc_diagnostics_str"] = (
    diag["admc_diagnostics"]
    .astype("string")
    .fillna("")
)

for code, label in diagnostic_map.items():
    diag[label] = diag["admc_diagnostics_str"].str.contains(
        rf"(^|\|)\s*{code}\s*(\||$)",
        regex=True
    ).astype(int)

diagnostic_long = diag.melt(
    id_vars=["episode_key", "lab_monitoring_quartile"],
    value_vars=list(diagnostic_map.values()),
    var_name="diagnostic_type",
    value_name="present"
)

diagnostic_rates = (
    diagnostic_long
    .groupby(
        ["lab_monitoring_quartile", "diagnostic_type"],
        observed=True
    )
    .agg(
        n=("present", "count"),
        rate=("present", "mean")
    )
    .reset_index()
)

display(diagnostic_rates)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=diagnostic_rates,
    x="lab_monitoring_quartile",
    y="rate",
    hue="diagnostic_type"
)

plt.xlabel("Laboratory monitoring intensity quartile")
plt.ylabel("Proportion of patients")
plt.title("Diagnostic procedures by laboratory monitoring intensity")
plt.xticks(rotation=30)
plt.legend(title="Diagnostic type", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT: ANTIMICROBIAL TREATMENT INTENSITY BY MONITORING QUARTILE
# ============================================================

antimic_map = {
    "0": "No antibiotic treatment needed",
    "1": "First line monotherapy or polytherapy (if only polytherapy was indicated)",
    "2": "First line polytherapy preferred over monotherapy",
    "3": "Second line antimicrobial therapy or first line antibiotics + antiviral/antifungal agents",
    "4": "Second/third line antibiotics + antiviral/antifungal agents, maximal polytherapy +/- antiviral and antifungal agents",
}

antimic = df[
    ["episode_key", "lab_monitoring_quartile", "admc_antimicrIntensity"]
].copy()

antimic["admc_antimicrIntensity_str"] = (
    antimic["admc_antimicrIntensity"]
    .astype("string")
    .fillna("")
)

for code, label in antimic_map.items():
    antimic[label] = antimic["admc_antimicrIntensity_str"].str.contains(
        rf"(^|\|)\s*{code}\s*(\||$)",
        regex=True
    ).astype(int)

antimic_long = antimic.melt(
    id_vars=["episode_key", "lab_monitoring_quartile"],
    value_vars=list(antimic_map.values()),
    var_name="antimic_int_type",
    value_name="present"
)

antimic_rates = (
    antimic_long
    .groupby(
        ["lab_monitoring_quartile", "antimic_int_type"],
        observed=True
    )
    .agg(
        n=("present", "count"),
        rate=("present", "mean")
    )
    .reset_index()
)

display(antimic_rates)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=antimic_rates,
    x="lab_monitoring_quartile",
    y="rate",
    hue="antimic_int_type"
)

plt.xlabel("Laboratory monitoring intensity quartile")
plt.ylabel("Proportion of patients")
plt.title("Antimicrobial treatment intensity by laboratory monitoring intensity")
plt.xticks(rotation=30)
plt.legend(title="Antimicrobial treatment intensity", bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT: ADMISSION MEWS BY MONITORING QUARTILE
# ============================================================

if "adm_mews" in df.columns:

    plt.figure(figsize=(8, 5))

    sns.boxplot(
        data=df,
        x="lab_monitoring_quartile",
        y="adm_mews"
    )

    plt.xlabel("Laboratory monitoring intensity quartile")
    plt.ylabel("MEWS at admission")
    plt.title("Admission severity by laboratory monitoring intensity")

    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# OUTCOME RATES BY MONITORING QUARTILE
# ============================================================

outcome_rate_rows = []

for outcome in [
    "composite_inhospital_or_palliative",
    "mortality_90d_post_discharge",
]:

    if outcome not in df.columns:
        continue

    tmp = df[
        ["lab_monitoring_quartile", outcome]
    ].dropna().copy()

    rate_df = (
        tmp
        .groupby("lab_monitoring_quartile", observed=True)[outcome]
        .agg(
            n="count",
            events="sum",
            event_rate="mean"
        )
        .reset_index()
    )

    rate_df["outcome"] = outcome
    outcome_rate_rows.append(rate_df)

monitoring_outcome_rates = pd.concat(
    outcome_rate_rows,
    ignore_index=True
)

display(monitoring_outcome_rates)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=monitoring_outcome_rates,
    x="lab_monitoring_quartile",
    y="event_rate",
    hue="outcome"
)

plt.xlabel("Laboratory monitoring intensity quartile")
plt.ylabel("Outcome rate")
plt.title("Outcome rates by laboratory monitoring intensity")
plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DYNAMIC FEATURE OBSERVABILITY
# ============================================================

dynamic_availability = []

for col in dynamic_cols:

    dynamic_availability.append({
        "feature": col,
        "available_pct": 100 * (df[col] == 1).mean()
    })

dynamic_availability = pd.DataFrame(dynamic_availability)

display(
    dynamic_availability
    .sort_values("available_pct")
    .head(30)
)

In [ ]:
# ============================================================
# TRAJECTORY OBSERVABILITY SUMMARY
# ============================================================

trajectory_summary = []

for biomarker in CORE_BIOMARKERS:

    row = {
        "biomarker": biomarker,

        "mean_0_3_available_pct":
            df[f"{biomarker}_mean_0_3"].notna().mean() * 100,

        "mean_3_7_available_pct":
            df[f"{biomarker}_mean_3_7"].notna().mean() * 100,

        "delta_0_3_available_pct":
            df[f"{biomarker}_delta_0_3"].notna().mean() * 100,

        "delta_3_7_available_pct":
            df[f"{biomarker}_delta_3_7"].notna().mean() * 100,

        "delta_0_7_available_pct":
            df[f"{biomarker}_delta_0_7"].notna().mean() * 100,
    }

    trajectory_summary.append(row)

trajectory_summary = pd.DataFrame(trajectory_summary)

display(
    trajectory_summary
    .sort_values("delta_3_7_available_pct", ascending=False)
)

In [ ]:
# ============================================================
# FINAL INTERPRETATION
# ============================================================

print("=" * 80)
print("04 FINAL MISSINGNESS ANALYSIS SUMMARY")
print("=" * 80)

print("\nDataset")
print("Rows:", df.shape[0])
print("Patients:", df["pubID"].nunique())
print("Episodes:", df["episode_key"].nunique())

print("\nLab availability")
print(df["has_any_lab_0_7"].value_counts(dropna=False))

print("\nMost missing features")
display(
    overall_missingness.head(15)
)

print("\nFeatures whose missingness is associated with composite outcome:")
print(
    "Nominal:",
    missing_tests_composite["significant_nominal"].sum(),
    "/",
    len(missing_tests_composite)
)
print(
    "FDR:",
    missing_tests_composite["significant_fdr"].sum(),
    "/",
    len(missing_tests_composite)
)

print("\nFeatures whose missingness is associated with 90-day mortality:")
print(
    "Nominal:",
    missing_tests_90d["significant_nominal"].sum(),
    "/",
    len(missing_tests_90d)
)
print(
    "FDR:",
    missing_tests_90d["significant_fdr"].sum(),
    "/",
    len(missing_tests_90d)
)

print("\nTop monitoring-complexity correlations")
display(
    monitoring_complexity_corr
    .sort_values("spearman_rho", ascending=False)
    .head(15)
)

print("\nInterpretation")
print("""
Missingness and laboratory trajectory observability are not random.

Laboratory availability, repeated testing, and monitoring intensity appear to
reflect clinically meaningful aspects of the hospitalization episode, including:
- hospitalization complexity
- physician monitoring intensity
- acute illness severity
- patient instability
- outcome risk

For this reason:
- n_days_* variables are retained;
- has_dynamic_* indicators are retained;
- global monitoring intensity features are retained;
- missingness is interpreted as clinically informative;
- tree-based models may exploit missingness structure directly;
- linear models should use conservative imputation strategies with missingness indicators.
""")

In [ ]:
# ============================================================
# SAVE OUTPUTS
# ============================================================

overall_missingness.to_excel(
    OUTPUT_04 / "04_missingness_overall.xlsx",
    index=False
)

pivot_missing_by_composite.to_excel(
    OUTPUT_04 / "04_missingness_by_composite_outcome.xlsx"
)

pivot_missing_by_90d.to_excel(
    OUTPUT_04 / "04_missingness_by_90d_mortality.xlsx"
)

missing_tests_composite.to_excel(
    OUTPUT_04 / "04_missingness_significance_composite.xlsx",
    index=False
)

missing_tests_90d.to_excel(
    OUTPUT_04 / "04_missingness_significance_90d.xlsx",
    index=False
)

dynamic_availability.to_excel(
    OUTPUT_04 / "04_dynamic_feature_availability.xlsx",
    index=False
)

trajectory_summary.to_excel(
    OUTPUT_04 / "04_trajectory_observability_summary.xlsx",
    index=False
)

monitoring_complexity_corr.to_excel(
    OUTPUT_04 / "04_monitoring_complexity_correlations.xlsx",
    index=False
)

monitoring_quartile_summary.to_excel(
    OUTPUT_04 / "04_monitoring_quartile_summary.xlsx",
    index=False
)

monitoring_quartile_compact.to_excel(
    OUTPUT_04 / "04_monitoring_quartile_compact.xlsx",
    index=False
)

monitoring_outcome_rates.to_excel(
    OUTPUT_04 / "04_monitoring_outcome_rates.xlsx",
    index=False
)

diagnostic_rates.to_excel(
    OUTPUT_04 / "04_diagnostic_procedures_by_monitoring_quartile.xlsx",
    index=False
)

antimic_rates.to_excel(
    OUTPUT_04 / "04_antimic_tx_intensity_by_monitoring_quartile.xlsx",
    index=False
)

print("All missingness and monitoring intensity outputs saved.")